# Ford Car Price Prediction

This notebook builds a **Linear Regression** model to predict the resale price
of used Ford cars based on attributes such as model, year, mileage, fuel type,
transmission, engine size, tax, and mpg.

**Workflow**
1. Load and inspect the dataset
2. Exploratory Data Analysis (EDA)
3. Feature encoding (One-Hot Encoding vs. Label Encoding)
4. Feature scaling
5. Train/test split and model training
6. Model evaluation (R² and Adjusted R²)
7. Compare encoding strategies

**Dataset:** [Ford Car Price Prediction](https://www.kaggle.com/datasets/adhurimquku/ford-car-price-prediction) (`ford.csv`)


## 1. Import Libraries

In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model selection & evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

# Consistent plot styling
sns.set_style("whitegrid")


## 2. Load the Dataset

Update `DATA_PATH` below to point to your local copy of `ford.csv`
(e.g. after downloading it from Kaggle or placing it in a `data/` folder).


In [ ]:
DATA_PATH = "data/ford.csv"  # <-- update this path if needed

df = pd.read_csv(DATA_PATH)
df.head()


## 3. Initial Data Inspection

In [ ]:
# Data types, non-null counts, and memory usage
df.info()


In [ ]:
# Summary statistics for numeric columns
df.describe()


In [ ]:
# Check for missing values in each column
df.isnull().sum()


## 4. Exploratory Data Analysis (EDA)

### 4.1 Price Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of Car Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.show()


### 4.2 Correlation Between Numeric Features

In [ ]:
# Numeric correlation matrix
df.corr(numeric_only=True)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


### 4.3 Price vs. Year

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="year", y="price")
plt.xticks(rotation=90)
plt.title("Price Distribution by Year")
plt.show()


### 4.4 Price vs. Mileage

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="mileage", y="price")
plt.title("Price vs. Mileage")
plt.show()


### 4.5 Price vs. Engine Size

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="engineSize", y="price")
plt.title("Price by Engine Size")
plt.show()


### 4.6 Price vs. Transmission

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="transmission", y="price")
plt.title("Price by Transmission Type")
plt.show()


### 4.7 Price vs. Fuel Type

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="fuelType", y="price")
plt.title("Price by Fuel Type")
plt.show()


### 4.8 Price vs. Model

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="model", y="price")
plt.xticks(rotation=90)
plt.title("Price by Car Model")
plt.show()


### 4.9 Price vs. Tax and MPG

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="tax", y="price")
plt.xticks(rotation=90)
plt.title("Price by Tax Band")
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="mpg", y="price")
plt.xticks(rotation=90)
plt.title("Price by MPG")
plt.show()


## 5. Split Features and Target

We separate the target column (`price`) from the feature set. Two parallel
feature sets are prepared so we can compare **One-Hot Encoding** against
**Label Encoding** for the categorical columns later on.


In [ ]:
X = df.drop(columns=["price"])
y = df["price"]


## 6. Approach A — One-Hot Encoding

Categorical columns (`transmission`, `fuelType`, `model`) are converted into
dummy/indicator columns. `drop_first=True` avoids the dummy-variable trap by
dropping one category per column.


In [ ]:
X_ohe = pd.get_dummies(X, columns=["transmission", "fuelType", "model"], drop_first=True)

# Convert boolean dummy columns to integers (0/1) for consistency
X_ohe = X_ohe.astype(int)
X_ohe.head()


### 6.1 Feature Scaling (One-Hot Encoded Set)

In [ ]:
numeric_cols = ["year", "mileage", "tax", "mpg", "engineSize"]

scaler_ohe = StandardScaler()
X_ohe[numeric_cols] = scaler_ohe.fit_transform(X_ohe[numeric_cols])
X_ohe.head()


## 7. Approach B — Label Encoding

As an alternative, each categorical column is label-encoded (mapped to
integer codes) instead of one-hot encoded, then all features are scaled.


In [ ]:
X_label = X.copy()

categorical_cols = ["model", "transmission", "fuelType"]
for col in categorical_cols:
    encoder = LabelEncoder()
    X_label[col] = encoder.fit_transform(X_label[col])

X_label.head()


### 7.1 Feature Scaling (Label Encoded Set)

In [ ]:
scaler_label = StandardScaler()
X_label[X_label.columns] = scaler_label.fit_transform(X_label[X_label.columns])
X_label.head()


## 8. Train & Evaluate — Model A (One-Hot Encoded Features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y, test_size=0.20, random_state=42
)

model_ohe = LinearRegression()
model_ohe.fit(X_train, y_train)


In [ ]:
y_pred_ohe = model_ohe.predict(X_test)


In [ ]:
r2_ohe = r2_score(y_test, y_pred_ohe)
print("R2 (One-Hot Encoding):", r2_ohe)


In [ ]:
# Adjusted R2 penalizes for the number of predictors, giving a fairer
# comparison between the two encoding approaches (which have different
# feature counts).
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2_ohe = 1 - (1 - r2_ohe) * (n - 1) / (n - p - 1)
print("Adjusted R2 (One-Hot Encoding):", adjusted_r2_ohe)


## 9. Train & Evaluate — Model B (Label Encoded Features)

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_label, y, test_size=0.20, random_state=42
)

model_label = LinearRegression()
model_label.fit(X2_train, y2_train)


In [ ]:
y_pred_label = model_label.predict(X2_test)


In [ ]:
r2_label = r2_score(y2_test, y_pred_label)
print("R2 (Label Encoding):", r2_label)


In [ ]:
n2 = X2_test.shape[0]
p2 = X2_test.shape[1]
adjusted_r2_label = 1 - (1 - r2_label) * (n2 - 1) / (n2 - p2 - 1)
print("Adjusted R2 (Label Encoding):", adjusted_r2_label)


## 10. Results Comparison

| Encoding Strategy | R² | Adjusted R² |
|---|---|---|
| One-Hot Encoding | see output above | see output above |
| Label Encoding | see output above | see output above |

Run the notebook end-to-end to populate the actual scores for your data.
One-hot encoding generally works better for linear models with unordered
categorical features (like `model` and `fuelType`), since label encoding
imposes an artificial ordinal relationship between categories.
